In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

MODEL='gpt-6-luna'

#### 구조화된 출력
- json 형식
- 결과값을 json 으로 받기 위해 json schema or pydantic 사용

In [2]:
# 사용자의 자기소개에서 이름과 나이를 추출

from openai import OpenAI
import json 

client = OpenAI()

response = client.responses.create(
    model=MODEL,
    input="안녕하세요. 저는 김민수이고 23살입니다.",
    text={
        "format":{
            "type":"json_schema",
            "name":"user_info",
            "strict":True,  # 모든 객체의 필드를 required 에 포함
            "schema":{
                "type":"object",
                "properties":{
                    "name":{
                        "type":"string"
                    },
                    "age":{
                        "type":"integer"
                    }
                },
                "required":["name","age"],
                "additionalProperties":False # 정의하지 않은 필드의 반환 제한

            }
        }
    }
)
result = json.loads(response.output_text)
print(result)

{'name': '김민수', 'age': 23}


In [3]:
# 상품 정보에서 문자열,숫자,불리언 추출
# {'name':'무선키보드','price':45000,'rating':4.7,'in_stock':True}
response = client.responses.create(
    model=MODEL,
    input="상품명은 무선 키보드이고, 가격은 45000원입니다. 재고는 현재 있으며, 평점은 4,7점입니다.",
    text={
        "format":{
            "type":"json_schema",
            "name":"product_info",
            "strict":True,  # 모든 객체의 필드를 required 에 포함
            "schema":{
                "type":"object",
                "properties":{
                    "name":{
                        "type":"string"
                    },
                    "price":{
                        "type":"integer" # 정수만 허용
                    },
                    "rating":{
                        "type":"number" # 정수, 실수 다 허용
                    },
                    "in_stock":{
                        "type":"boolean"
                    }
                },
                "required":["name","price","rating","in_stock"],
                "additionalProperties":False # 정의하지 않은 필드의 반환 제한

            }
        }
    }
)

result = json.loads(response.output_text)
print(result)

{'name': '무선 키보드', 'price': 45000, 'rating': 4.7, 'in_stock': True}


In [6]:
# 고객문의를 분석하여 문의 유형과 중요도로 응답
# {'category':'delivery','priority':'high','summary':'요청사항'}
# category : payment, delivery, refund, etc
# priority : high, medium, low



response = client.responses.create(
    model=MODEL,
    input="결제는 완료했는데 상품이 배송되지 않고 있습니다. 빠르게 확인해주세요",
    text={
        "format":{
            "type":"json_schema",
            "name":"customer_inquiry",
            "strict":True,  # 모든 객체의 필드를 required 에 포함
            "schema":{
                "type":"object",
                "properties":{
                    "category":{
                        "type":"string",
                        "enum":[
                            "payment", "delivery", "refund", "etc",
                        ],
                    },
                    "priority":{
                        "type":"string",
                        "enum":[
                            "high", "medium", "low",
                        ],
                    },
                    "summary":{
                        "type":"string" 
                    },

                },
                "required":["category","priority","summary"],
                "additionalProperties":False # 정의하지 않은 필드의 반환 제한

            }
        }
    }
)

result = json.loads(response.output_text)
print(result)






{'category': 'delivery', 'priority': 'high', 'summary': '결제는 완료했으나 상품이 배송되지 않아 빠른 확인을 요청하셨습니다.'}


In [ ]:
input = """제품을 구매한지 일주일이 지났는데 아직 배송되지 않았습니다.
고객센터에 여러 번 문의했지만 답변도 받지 못했습니다.
환불을 요청합니다.
"""

# {sentiment:navative, intent:refund, keywords:[배송지연, 고객센터 미응답, 환불요청],summary:''}
# sentiment: positive, neutral, negative
# intent : delivery, refund, exchange, complain, etc
# keywords: type: array, items:{type:string} => 모델이 알아서 추출함, 부여 X
# summary : ''


response = client.responses.create(
    model=MODEL,
    input=input,
    text={
        "format":{
            "type":"json_schema",
            "name":"sentiment_analysis",
            "strict":True,  # 모든 객체의 필드를 required 에 포함
            "schema":{
                "type":"object",
                "properties":{
                    "sentiment":{
                        "type":"string",
                        "enum":[
                            "positive", "neutral", "negative"
                        ]
                    },
                    "intent":{
                        "type":"string",
                        "enum":[
                            "delivery", "refund", "exchange", "complain", "etc"
                        ]
                    },
                    "keywords":{
                        "type":"array",
                        "items":{
                            "type":"string"
                        } 
                    },                    
                    "summary":{
                        "type":"string" 
                    },

                },
                "required":["sentiment","intent","keywords","summary"],
                "additionalProperties":False # 정의하지 않은 필드의 반환 제한

            }
        }
    }
)

result = json.loads(response.output_text)
# print(result)


{'sentiment': 'negative', 'intent': 'refund', 'keywords': ['배송 지연', '일주일 미배송', '고객센터 무응답', '환불 요청'], 'summary': '제품을 구매한 지 일주일이 지났지만 배송되지 않았고, 고객센터에 여러 차례 문의했으나 답변을 받지 못해 환불을 요청합니다.'}


In [ ]:
result = json.loads(response.output_text)
print(json.dumps(result, ensure_ascii=False,indent=4))

{
    "sentiment": "negative",
    "intent": "refund",
    "keywords": [
        "배송 지연",
        "일주일 미배송",
        "고객센터 무응답",
        "환불 요청"
    ],
    "summary": "제품을 구매한 지 일주일이 지났지만 배송되지 않았고, 고객센터에 여러 차례 문의했으나 답변을 받지 못해 환불을 요청합니다."
}


In [ ]:
# 계약서 내용을 분석하여 계약 당사자, 계약 기간, 계약 금액 등을 추출
# {'contract_date':2026-03-01, 
# 'parties':['주식회사 ABC','김철수'], 
# 'period':{start_date:2026-03-01, end_date:2027-02-28}, 
# 'amount':5000000}
input = """
계약서 내용:

계약 당사자는 주식회사 ABC와 김철수이다.
계약 체결일은 2026년 3월 1일이다.
계약 기간은 2026년 3월 1일부터 2027년 2월 28일까지이다.
계약 금액은 5,000,000이다.
"""

response = client.responses.create(
    model=MODEL,
    input=input,
    text={
        "format":{
            "type":"json_schema",
            "name":"contract_info",
            "strict":True,  # 모든 객체의 필드를 required 에 포함
            "schema":{
                "type":"object",
                "properties":{
                    "contract_date":{
                        "type":"string",
                    },
                    "parties":{
                        "type":"array",
                        "items":{
                            "type":"string"
                        }
                    },
                    "period":{
                        "type":"object",
                        "properties":{
                            "start_date":{
                                "type":"string",
                            },
                            "end_date":{
                                "type":"string",
                            },

                        },
                                                 "required":["start_date","end_date"],
                         "additionalProperties":False   
                    },                    
                    "amount":{
                        "type":"integer" 
                    },

                },
                "required":["contract_date","parties","period","amount"],
                "additionalProperties":False # 정의하지 않은 필드의 반환 제한

            }
        }
    }
)

result = json.loads(response.output_text)
print(json.dumps(result, ensure_ascii=False,indent=4))

BadRequestError: Error code: 400 - Invalid schema for response_format 'contract_info': ['start_date', 'end_date'] is not of type 'object', 'boolean'.